### exp 01
This experiment is done before fine-tuning the model to analyse and evaluate the model
output. Later after fine tuning, we'll compare results with baseline model outputs
to determine the improvement. So this is basically the starting point.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1329.69it/s]


In [3]:
# baseline probes
baseline_examples = [
    {
        "id": "baseline_001",
        "input": "I forgot my password and can't access my account.",
        "intent": "password_reset",
        "priority": "medium",
    },
    {
        "id": "baseline_002",
        "input": "Someone changed the email address on my account.",
        "intent": "account_compromise",
        "priority": "high",
    },
    {
        "id": "baseline_003",
        "input": "I was charged twice for the same subscription.",
        "intent": "billing_issue",
        "priority": "medium",
    },
    {
        "id": "baseline_004",
        "input": "Please add dark mode to the application.",
        "intent": "feature_request",
        "priority": "low",
    },
    {
        "id": "baseline_005",
        "input": "I want to close my account permanently.",
        "intent": "account_closure",
        "priority": "low",
    },
]

In [11]:
def  generate_response(model,tokenizer,user_text : str):
    messages = [
        {
            'role':'user',
            'content':user_text,
        }
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt'
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False #reproducibility
    )
    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [ ]:
import json
example_copy = baseline_examples.copy()
for example in example_copy:
    response = generate_response(
        model,
        tokenizer,
        example['input']
    )

    print('-'*100)
    print("INPUT: ", example['input'])
    print("INTENT: ", example['intent'])
    print("MODEL: ", response)
    example['response'] = response



----------------------------------------------------------------------------------------------------
INPUT:  I forgot my password and can't access my account.
INTENT:  password_reset
MODEL:  system
You are a helpful AI assistant named SmolLM, trained by Hugging Face
user
I forgot my password and can't access my account.
assistant
I'm sorry for the confusion, but as a chatbot, I don't have the ability to access or manage your account. I'm here to assist you with any inquiries or issues you might have. If you need help with something else, feel free to ask.
----------------------------------------------------------------------------------------------------
INPUT:  Someone changed the email address on my account.
INTENT:  account_compromise
MODEL:  system
You are a helpful AI assistant named SmolLM, trained by Hugging Face
user
Someone changed the email address on my account.
assistant
I'm sorry for the confusion, but as a chatbot, I don't have the ability to access or manage personal acc

In [15]:
with open('predictions.jsonl', 'w', encoding="utf-8") as f:
    for item in example_copy:
        f.write(json.dumps(item) + '\n')

### outcomes

we have some outputs generated by our baseline model, it does not produce desired output format, actions are not correct as well.  we'll use same mode, same input, same generation configuration after fine-tuning to analyze the performance improvement.